# Data Mining II — Group Project 2025/2026
## Reach for Change: Predicting Donor Response to Optimize Outreach for Social Good

**Group 34**

> Civic Support Alliance (CSA) wants to contact fewer people but the *right* people.
> Our task is to build a binary classifier that predicts whether a person will donate
> if contacted, using historical campaign, demographic, and behavioural data.


---
## Group Member Contribution

| Member | Student Number |
|---|---|
| *Name 1* | *xxxxx* |
| *Name 2* | *xxxxx* |
| *Name 3* | *xxxxx* |
| *Name 4* | *xxxxx* |

---
## Abstract

> **TODO — 200 to 300 words.** Write this *last*, once results are final.
>
> Cover: the context (CSA's outreach problem), project goals (binary
> classification of donor response), what we did (EDA, preprocessing,
> modelling with at least 4 algorithms, optimization, Kaggle submission,
> open-ended analysis), main results (best model, F1 score on holdout and
> Kaggle), and the conclusions drawn from them.


---
# I. Introduction

## Overview and main goals

The **Civic Support Alliance (CSA)** has historically contacted every individual in
its database during fundraising campaigns. This blanket approach produces donor
fatigue and diminishing returns. The organisation now wants to contact only
individuals who are most likely to respond positively — improving both operational
efficiency and donor experience.

Our goal is therefore to build a **binary classifier** that answers:
*"Will this person donate if contacted?"* — where the positive class (`TARGET_B = 1`)
denotes a donor who responded to last year's campaign.

## Process overview

1. **Data Exploration & Preprocessing** — understand the data, handle missing values,
   encode categoricals, treat outliers, and engineer features.
2. **Modelling & Optimization** — train and compare at least four algorithms covered
   in class, using a consistent assessment strategy, and optimize the best candidates.
3. **Deployment** — generate predictions on `donors_test.csv` and submit to Kaggle.
4. **Open-Ended Analysis** — extend the work with additional insights
   (e.g. feature importance, error analysis, calibration).

## Model assessment strategy

> **TODO — state the approach clearly.** We plan to use **stratified k-fold
> cross-validation** (k = TODO) on the training set, preserving the class
> distribution of `TARGET_B`. A separate stratified **hold-out split**
> (TODO %) will be reserved as an internal "unseen" test to corroborate CV
> results before the Kaggle submission. The primary metric is **binary F1
> score**, matching the Kaggle leaderboard; supporting metrics
> (accuracy, precision, recall, ROC-AUC) are reported to aid interpretation.


---
## Setup — imports and configuration

All imports are grouped here so the notebook runs from start to finish in order.
Only packages from **vanilla scikit-learn** and standard scientific Python are
used (XGBoost / LightGBM / CatBoost / AutoML packages are explicitly off-limits
per the project guidelines).


In [ ]:
# Standard library
import os
import warnings
from pathlib import Path

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn — preprocessing
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# scikit-learn — models (at least 4 covered in class)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# scikit-learn — metrics & feature selection
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    mutual_info_classif,
    RFE,
)

# Notebook configuration
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 34  # group number, used for all reproducible splits


### Data paths — works locally and in Google Colab

The block below detects whether we are running in Colab. If so, it mounts Google
Drive and points `DATA_PATH` to the shared project folder; otherwise it falls
back to the local repository's `project_data/` folder. This lets the same
notebook run in both environments without edits.


In [ ]:
try:
    # Running in Google Colab?
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DATA_PATH = Path("/content/drive/MyDrive/DM2-Project/project_data")
    IN_COLAB = True
except ImportError:
    DATA_PATH = Path("project_data")
    IN_COLAB = False

print(f"In Colab: {IN_COLAB}")
print(f"Data path: {DATA_PATH.resolve()}")
assert DATA_PATH.exists(), f"Data folder not found at {DATA_PATH}"


In [ ]:
TRAIN_FILE = DATA_PATH / "donors_train.csv"
TEST_FILE = DATA_PATH / "donors_test.csv"
SAMPLE_SUB_FILE = DATA_PATH / "sample_submission.csv"

train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE)
sample_submission = pd.read_csv(SAMPLE_SUB_FILE)

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")
train.head()


---
# II. Data Exploration and Preprocessing

**Goal:** understand what we are working with and prepare clean, model-ready data.

This section is worth **4 / 20** of the final grade. According to the guidelines,
we should *extract meaningful insights relevant to the problem* and *avoid
unnecessary visualizations*. Every chart we keep should drive a decision.


## 2.1 Dataset overview

We start by inspecting the dimensionality, column types, and a sample of the data
to get a first sense of what each feature looks like.


In [ ]:
train.info()


In [ ]:
train.describe(include="all").T


## 2.2 Target variable

`TARGET_B` is our binary target: 1 if the person donated in last year's campaign,
0 otherwise. We check its distribution to know whether we need to handle class
imbalance in our modelling strategy.


In [ ]:
target_counts = train["TARGET_B"].value_counts(dropna=False)
target_share = train["TARGET_B"].value_counts(normalize=True, dropna=False)
print(target_counts)
print()
print(target_share.round(4))

fig, ax = plt.subplots(figsize=(4, 3))
target_counts.plot(kind="bar", ax=ax)
ax.set_title("Target distribution (TARGET_B)")
ax.set_xlabel("TARGET_B")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


> **TODO — key insight.** Comment on the class balance and how it will
> influence our choice of metric (F1) and any resampling / class_weight handling.


## 2.3 Missing values

We quantify missingness per column to decide between dropping, imputing
(mean / median / mode / KNN), or encoding "missing" as its own category.


In [ ]:
missing = (
    train.isna()
    .mean()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
    .rename("missing_ratio")
    .to_frame()
)
missing


## 2.4 Feature typing

We separate features into numerical and categorical groups, and identify the
identifier (`CONTROL_NUMBER`) and target (`TARGET_B`) so they are never fed as
features to the model.


In [ ]:
ID_COL = "CONTROL_NUMBER"
TARGET = "TARGET_B"

features = [c for c in train.columns if c not in {ID_COL, TARGET}]
numerical_features = train[features].select_dtypes(include="number").columns.tolist()
categorical_features = train[features].select_dtypes(exclude="number").columns.tolist()

print(f"# numerical:   {len(numerical_features)}")
print(f"# categorical: {len(categorical_features)}")


## 2.5 Univariate analysis — numerical features

We look at distributions and basic statistics to spot skewness, outliers, and
features that may need transformation or scaling.

> **TODO — keep only the plots that reveal something we act on.**


In [ ]:
# TODO: distributions / boxplots of key numerical features
# (e.g. LIFETIME_GIFT_AMOUNT, RECENT_AVG_GIFT_AMT, DONOR_AGE, ...)


## 2.6 Univariate analysis — categorical features

We inspect the cardinality and distribution of each categorical feature to
plan our encoding strategy (one-hot vs. ordinal vs. target encoding).


In [ ]:
# TODO: value counts / bar plots for URBANICITY, SES, HOME_OWNER,
#   DONOR_GENDER, RECENCY_STATUS_96NK, ...


## 2.7 Bivariate analysis — relationships with the target

We look at how each feature relates to `TARGET_B` to build intuition about
which variables carry predictive signal.


In [ ]:
# TODO: grouped means / boxplots / cross-tabs vs. TARGET_B


## 2.8 Correlations between numerical features

A correlation matrix helps spot redundant features that we may want to drop
(for linear models especially) or flag for feature selection.


In [ ]:
# TODO: correlation heatmap, flag highly correlated pairs


## 2.9 Preprocessing plan

Based on the findings above, we apply the following transformations:

1. **Drop the identifier** `CONTROL_NUMBER` from the feature matrix.
2. **Missing values:** *TODO — state the imputation strategy per column group.*
3. **Outliers:** *TODO — clipping / winsorising / leaving them, with rationale.*
4. **Categorical encoding:** *TODO — one-hot for nominal, ordinal for ordered codes.*
5. **Scaling:** *TODO — StandardScaler for distance-based and linear models.*
6. **Feature engineering:** *TODO — any ratios/bins we add (e.g. gift per promotion).*

All of the above is wrapped in a scikit-learn `Pipeline` + `ColumnTransformer` so
that transformations are fit on training folds only (avoiding leakage) and
applied consistently at inference time.


In [ ]:
# TODO: build numeric_pipe, categorical_pipe, and a ColumnTransformer
# preprocessor that we can plug into every model below.


---
# III. Binary Classification & Optimization

Worth **6 / 20** — the single heaviest section. It breaks into five components
per the guidelines:

1. Additional preprocessing
2. Feature selection strategy and implementation
3. Modelling approach (assessment strategy + ≥ 4 algorithms from class)
4. Performance assessment (choice of metric + interpretation)
5. Model optimization


## 3.1 Train / validation split + assessment strategy

We reserve a stratified hold-out (*TODO — e.g. 20%*) as an internal "unseen"
test set. All model selection and optimization is done on the remaining training
data using **stratified k-fold cross-validation** (*TODO — k = 5 / 10*).


In [ ]:
X = train.drop(columns=[ID_COL, TARGET])
y = train[TARGET]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_holdout: {X_holdout.shape}, y_holdout: {y_holdout.shape}")


## 3.2 Performance metric

Per the Kaggle leaderboard, our primary metric is the **binary F1 score**,
which balances precision and recall — the right trade-off when the positive
class matters and imbalance is present. We additionally track accuracy,
precision, recall, and ROC-AUC for diagnostic purposes.


In [ ]:
SCORING = {
    "f1": "f1",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
}
PRIMARY_METRIC = "f1"


## 3.3 Feature selection

We compare feature selection strategies (filter, wrapper, embedded) and
keep the subset that maximises CV F1 for our best baseline model.

> **TODO — document the strategy chosen and why.**


In [ ]:
# TODO: SelectKBest / RFE / feature importance from a RandomForest


## 3.4 Baseline models

We fit **at least four** algorithms covered in class using the same CV splits
so results are directly comparable. Models considered: Logistic Regression,
Decision Tree, K-Nearest Neighbours, Gaussian Naive Bayes, Random Forest,
Gradient Boosting (sklearn), and MLP.


In [ ]:
# TODO: replace `preprocessor` below with the ColumnTransformer built in 2.9.
# Kept as a stub so the cell documents the intended structure.

# baseline_models = {
#     "LogReg":        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
#     "DecisionTree":  DecisionTreeClassifier(random_state=RANDOM_STATE),
#     "KNN":           KNeighborsClassifier(),
#     "GaussianNB":    GaussianNB(),
#     "RandomForest":  RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
#     "GradBoost":     GradientBoostingClassifier(random_state=RANDOM_STATE),
#     "MLP":           MLPClassifier(random_state=RANDOM_STATE, max_iter=500),
# }
#
# baseline_results = {}
# for name, clf in baseline_models.items():
#     pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
#     scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
#     baseline_results[name] = {m: scores[f"test_{m}"].mean() for m in SCORING}
#
# baseline_df = pd.DataFrame(baseline_results).T.sort_values(PRIMARY_METRIC, ascending=False)
# baseline_df


## 3.5 Model comparison and discussion

> **TODO — interpret the baseline table.** Which models dominate on F1?
> Are there trade-offs between precision and recall? Does a simple model
> (LogReg) already cover most of the signal, or do ensembles win clearly?


## 3.6 Hyperparameter optimization

We pick the top **2–3** baselines and tune them with `GridSearchCV` (or
`RandomizedSearchCV` when the space is large), optimising binary F1 on the
same CV splits. After grid search, we refit the final model with the chosen
hyperparameters in a separate cell (so evaluators can re-run the notebook
without waiting for the search).


In [ ]:
# TODO: grid / randomised search for top models.
# REMEMBER: comment out the search cell before submission and fit the chosen
# hyperparameters in a separate cell, per the project guidelines.


## 3.7 Final model selection

> **TODO — state which model was chosen for deployment and why**, comparing
> CV F1, hold-out F1, variance across folds, training time, and interpretability.


In [ ]:
# TODO: fit the final model on X_train (or X_train + X_holdout, see below)
# and report hold-out metrics (F1, precision, recall, AUC, confusion matrix).


---
# IV. Deployment

We now take the final pipeline, refit it on **all** labelled data
(`X_train + X_holdout`) so that we do not waste information, and generate
predictions for the Kaggle test set. The output file follows the naming
convention `DM2_Group34_<tag>.csv`.


In [ ]:
# TODO: refit final_pipeline on full train data and predict on `test`

# final_pipeline.fit(X, y)
# test_ids = test[ID_COL]
# test_features = test.drop(columns=[ID_COL])
# predictions = final_pipeline.predict(test_features)
#
# submission = pd.DataFrame({ID_COL: test_ids, "TARGET_B": predictions})
# submission_path = Path("DM2_Group34_submission.csv")
# submission.to_csv(submission_path, index=False)
# print(f"Wrote {submission_path}  ({len(submission)} rows)")
# submission.head()


---
# V. Open-Ended Section

The guidelines explicitly note this section *"expects that the objectives set
go beyond what would reasonably be considered as adding or removing techniques
to your pipeline."* We therefore formulate clear, scoped objectives rather than
sprinkling extra tricks.

## Objectives (TODO — pick 1–2 and commit)

Candidate directions from the brief:

- **(a) Feature importance analysis** — permutation importance + SHAP-free
  inspection to explain which variables drive predictions and whether they
  align with domain intuition about donor behaviour.
- **(b) Error analysis** — characterise the false positives and false negatives
  to understand *who* we misclassify and what it would cost the CSA.
- **(c) Probability calibration** — inspect reliability diagrams and apply
  Platt scaling / isotonic regression to turn raw scores into trustworthy
  probabilities for outreach budgeting.
- **(d) Mini analytics interface** — a small function (or `ipywidgets` form)
  that takes a donor record and returns a probability + explanation.

> **Chosen objective(s):** *TODO*


In [ ]:
# TODO: implement the chosen open-ended analysis


## Results and key takeaways

> **TODO — summarise what we found, plainly, and tie it back to the CSA's
> real problem (contacting the *right* people, not everyone).**


---
## Conclusions

> **TODO** — wrap up: what worked, what didn't, limitations, and what we would
> do next with more time.